# H&N vs Lung – Phase 1: link the data

**Goal:** for every account that buys Rybrevant (DDD), know which HCPs treat there and whether they treat Lung, H&N or both.

```
Claims (MX + RX) ──dv1|dv2──► patient ──dx codes──► LUNG / HN
      │ NPI
      ▼
  DMN_CUST (NPI → JJ_ID)  ──►  OPAL `0` (HCP JJ_ID) ──► L2_L3 account
                                                              ▲
DDD jnj_id (L1 outlet) ──► OPAL `1` ─────────────────────────┘
ChargeMaster: national NSCLC / HNSCC share by period (no join, used as a check)
```
Sections: 1 load · 2 claims → patient / HCP labels · 3 DDD → OPAL account · 4 HCP → account · 5 account table · 6 ChargeMaster check

**Filters already applied upstream (per `RYB_HN_Data_Sources.csv`), so this notebook does NOT repeat them:**

| Table | Already filtered on |
|---|---|
| DDD (`cdp.fct_nonret_sls_trans_wkly`) | product name contains RYBREVANT or LAZCLUZE |
| Rx claims (`fct_link_int_lung_phrmcy_evnts`) | Rybrevant / Lazcluze NDCs, `claim_type = PAID`, `claim_status = S` |
| Mx claims (`fct_link_int_lung_mdcl_evnts`) | Rybrevant / Lazcluze NDCs, `service_date >= 2025-01-01` |
| OPAL (`d_provider_hierarchy_pd`) | relation type contains ONC |
| dmn_cust | `end_dt = 2099-12-31` (current record), `cust_stat_nm = Active` |

## 1. Load data

### What this cell does: load the data
- Connects to Spark and reads the 5 Dataiku datasets into Spark DataFrames:
  - `mx`: medical claims (Rybrevant infusions)
  - `rx`: pharmacy claims
  - `ddd`: DDD sales
  - `opal`: OPAL hierarchy
  - `cm`: ChargeMaster
- Also reads **`dmn_cust`**, the customer master used as the NPI → J&J ID bridge.
  - It is shown under project TA_HCP_UNIVERSER_VERIFY. If it is not shared into H&N_Project, write it as `"PROJECTKEY.dmn_cust"` (use that project's key) or share it first.
  - Set `DMN_CUST_NAME = ""` to skip it.
- Prints the row and column count of each table so you can see that everything loaded.

In [ ]:
import dataiku
from dataiku import spark as dkuspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
spark = SparkSession.builder.getOrCreate()

def load(name):
    return dkuspark.get_dataframe(spark, dataiku.Dataset(name))

mx   = load("MX_RYB_CLAIMS_copy")        # medical claims (infusions)
rx   = load("RYB_LINK_CLAIMS_copy")      # pharmacy claims
ddd  = load("RYB_DDD_Sales_copy")        # DDD sales, weekly, jnj_id = L1 outlet
opal = load("OPAL_Affiliation_Levels")   # HCP -> L1..L5 parents (wide)
cm   = load("ChargeMaster_WeeklyReporting_20261012_v2_copy")

# DMN_CUST: customer master, the NPI -> J&J ID bridge (npi_id -> jnj_id).
# If dmn_cust sits in another Dataiku project, use "PROJECTKEY.dmn_cust" (or share it into H&N_Project).
DMN_CUST_NAME = "dmn_cust"
dmn = load(DMN_CUST_NAME) if DMN_CUST_NAME else None

for name, df in [("mx", mx), ("rx", rx), ("ddd", ddd), ("opal", opal), ("chargemaster", cm), ("dmn_cust", dmn)]:
    if df is None:
        continue
    print(f"{name:13s} rows={df.count():>11,}  cols={len(df.columns)}")

### What this cell does: define small helper rules (nothing is calculated yet)
- `clean()`: trims spaces and turns empty text into null, so blanks don't count as real values.
- `LUNG_RE` / `HN_RE`: diagnosis code patterns.
  - Lung = codes starting with **C34**.
  - Head & Neck = **C00–C14, C30–C32, C76.0** (provisional; replace with Siddhant's list).
  - Metastasis codes (C77–C79) and chemo-visit codes (Z51.1x) are ignored on purpose, because they don't tell where the cancer started.
- `dx_norm()`: puts the diagnosis text in one format (upper case, no dots, separated by `|`) so the patterns above match.
- `NDC_PRODUCT` / `product_expr()`: decides the product for each claim.
  - First the **NDC reference** from the data-sources file: Rybrevant, Rybrevant Faspro, Lazcluze.
  - Then the billing code: **J9061 = Rybrevant**, **J9062 = Faspro**.
  - Then the product name.
- `RYB_PRODUCTS`: Rybrevant + Faspro, the products whose sales we split Lung vs H&N.
  - Lazcluze is lung-only, so it is kept as lung evidence but not split.

In [ ]:
# helpers
def clean(c):
    """trim, empty string -> null"""
    c = F.trim(F.col(c).cast("string"))
    return F.when(c != "", c)

LUNG_RE = r"\|C34"                                  # lung primary
HN_RE   = r"\|(C0[0-9]|C1[0-4]|C3[0-2]|C760)"       # H&N primary (provisional - replace with Siddhant's list)
# C77-C79 (metastases) and Z51.1x (chemo encounter) are ignored on purpose

def dx_norm(c):
    """'|Z5111|C34.90|' -> '|Z5111|C3490|' (upper, no dots, pipe-wrapped)"""
    s = F.upper(F.regexp_replace(F.coalesce(F.col(c), F.lit("")), r"[.\s]", ""))
    return F.concat(F.lit("|"), F.regexp_replace(s, r"[;,]", "|"), F.lit("|"))

# NDC reference (from RYB_HN_Data_Sources.csv)
NDC_PRODUCT = {
    "57894050100": "RYBREVANT", "57894050101": "RYBREVANT",
    "57894051001": "RYBREVANT_FASPRO", "57894051401": "RYBREVANT_FASPRO",
    "57894051501": "RYBREVANT_FASPRO", "57894052201": "RYBREVANT_FASPRO",
    "57894008060": "LAZCLUZE", "57894008090": "LAZCLUZE", "57894024030": "LAZCLUZE", "57894024099": "LAZCLUZE",
}
ndc_map = F.create_map(*[F.lit(x) for kv in NDC_PRODUCT.items() for x in kv])
RYB_PRODUCTS = ["RYBREVANT", "RYBREVANT_FASPRO"]     # split Lung vs H&N; Lazcluze is lung-only and not split

def product_expr(hcpcs, ndc, name):
    """NDC reference first, then HCPCS (J9061 / J9062), then the product name."""
    nm = F.upper(F.coalesce(F.col(name), F.lit("")))
    by_code = (F.when(F.col(hcpcs) == "J9062", "RYBREVANT_FASPRO")
                .when(F.col(hcpcs) == "J9061", "RYBREVANT")) if hcpcs else F.lit(None).cast("string")
    by_name = (F.when(nm.contains("FASPRO"), "RYBREVANT_FASPRO").when(nm.contains("RYBREVANT"), "RYBREVANT")
                .when(nm.contains("LAZCLUZE"), "LAZCLUZE"))
    return F.coalesce(ndc_map[F.regexp_replace(F.col(ndc), "[^0-9]", "")], by_code, by_name)

## 2. Claims → patient label → HCP label
The claims are already filtered upstream (NDCs, PAID / status S for Rx, service date for Mx), so they are used as they come.
Only additions: HCP = rendering NPI else referring NPI (Mx), drop 0-quantity Mx lines (empty lines, not a business filter),
and one administration = patient + date + product (professional and facility lines of the same infusion count once).

### What this cell does: put the medical and pharmacy claims into one table
- **No business filters are repeated here.** Upstream already kept only Rybrevant/Lazcluze NDCs, PAID + status S pharmacy claims, and medical claims from 2025 on.
- **Medical claims (`mx`)**:
  - drops lines with quantity 0 (empty lines with no product, not a real administration);
  - doctor = `rendering_npi`, or `referring_npi` when rendering is empty.
- **Pharmacy claims (`rx`)**:
  - shows the `claim_type` / `claim_status` values as a check only (they should all be PAID / S);
  - doctor = `prescriber_npi`.
- Both are reshaped to the same columns: patient (`dv1|dv2`), date, doctor NPI, organization NPI, diagnosis, product.
- They are then stacked into one table `claims`.
- Every row gets a **claim_label** from its own diagnosis codes: LUNG, HN, BOTH, or NO_LUNG_HN_DX.
- Prints how many rows were kept from each table.

In [ ]:
mx_c = (mx.filter(F.coalesce(F.col("quantity"), F.lit(1.0)) > 0)
          .select(F.lit("MX").alias("source"),
                  F.col("mx_claim_id").alias("claim_id"),
                  F.concat_ws("|", "dv1", "dv2").alias("patient_id"),
                  F.to_date("service_date").alias("service_dt"),
                  F.coalesce(clean("rendering_npi"), clean("referring_npi")).alias("npi"),
                  clean("billing_npi").alias("org_npi"),
                  F.col("data_source").alias("setting"),
                  dx_norm("diagnosis_codes").alias("dx"),
                  product_expr("procedure_code", "ndc", "product_name").alias("product"),
                  "procedure_code", "ndc", "product_name", "quantity"))

rx.groupBy("claim_type", "claim_status").count().show()   # check only: upstream filter = PAID / S
rx_c = (rx.select(F.lit("RX").alias("source"), "claim_id",
                  F.concat_ws("|", "dv1", "dv2").alias("patient_id"),
                  F.to_date("fill_date").alias("service_dt"),
                  clean("prescriber_npi").alias("npi"),
                  clean("pharmacy_npi").alias("org_npi"),
                  F.col("pharmacy_channel").alias("setting"),
                  dx_norm("diagnosis_code").alias("dx"),
                  product_expr(None, "ndc", "product_name").alias("product"),   # no HCPCS on Rx
                  F.lit(None).cast("string").alias("procedure_code"), "ndc", "product_name", "quantity"))

claims = (mx_c.unionByName(rx_c)
          .withColumn("claim_label",
                      F.when(F.col("dx").rlike(LUNG_RE) & F.col("dx").rlike(HN_RE), "BOTH")
                       .when(F.col("dx").rlike(LUNG_RE), "LUNG")
                       .when(F.col("dx").rlike(HN_RE), "HN")
                       .otherwise("NO_LUNG_HN_DX"))
          .withColumn("month", F.trunc("service_dt", "month"))
          .cache())
print("mx kept:", mx_c.count(), "of", mx.count(), "| rx kept:", rx_c.count(), "of", rx.count())

### What this cell does: sanity checks on the claims (only displays results)
1. **Product check**: NDC × billing code × product name → product we assigned. Look for rows where they disagree, or where the product is empty (NDC not in the reference).
2. **Label check**: number of claim lines and patients that are Lung / H&N / both / neither, by source.
3. **Top 25 diagnosis codes**: shows which codes really appear, to confirm the Lung / H&N code lists are right.

In [ ]:
# Checks: product mapping vs product_name, label distribution, top diagnosis codes
claims.groupBy("source", "ndc", "procedure_code", "product_name", "product").count().orderBy(F.desc("count")).show(20, False)
claims.groupBy("source", "claim_label").agg(F.count("*").alias("lines"), F.countDistinct("patient_id").alias("patients")).show()
(claims.select(F.explode(F.split("dx", r"\|")).alias("code")).filter("code != ''")
       .groupBy("code").count().orderBy(F.desc("count")).show(25))

### What this cell does: label every patient, then every doctor
- **Patient label**: for each patient, count on how many different dates they had a Lung code and on how many an H&N code.
  - Only Lung → **LUNG**; only H&N → **HN**.
  - Both: if one side is ≥ 80% of the dates, the patient takes that label; otherwise **DUAL** (unclear).
- **Doctor label**: for each doctor NPI, count their Lung patients and H&N patients.
  - Lung only → **LUNG_ONLY**; H&N only → **HN_ONLY**; both kinds → **BOTH**.
  - Only DUAL patients → UNRESOLVED_DUAL; no cancer codes → NO_DX.
- Shows how many patients and doctors fall in each group. The `hcp` table is the **HCP universe**.

In [ ]:
# Patient label: count distinct dx dates per indication; both -> dominant if >= 80%, else DUAL
pt = (claims.groupBy("patient_id")
      .agg(F.countDistinct(F.when(F.col("dx").rlike(LUNG_RE), F.col("service_dt"))).alias("lung_dates"),
           F.countDistinct(F.when(F.col("dx").rlike(HN_RE), F.col("service_dt"))).alias("hn_dates")))
share = F.col("hn_dates") / (F.col("hn_dates") + F.col("lung_dates"))
patients = pt.withColumn("patient_label",
    F.when((F.col("lung_dates") > 0) & (F.col("hn_dates") == 0), "LUNG")
     .when((F.col("hn_dates") > 0) & (F.col("lung_dates") == 0), "HN")
     .when((F.col("hn_dates") > 0) & (share >= 0.8), "HN")
     .when((F.col("lung_dates") > 0) & (share <= 0.2), "LUNG")
     .when(F.col("hn_dates") > 0, "DUAL")
     .otherwise("UNKNOWN"))
patients.groupBy("patient_label").count().show()

# HCP label: distinct patients by label for each NPI on the claims
hcp = (claims.filter(F.col("npi").isNotNull()).select("npi", "patient_id").distinct()
       .join(patients, "patient_id")
       .groupBy("npi")
       .agg(F.countDistinct(F.when(F.col("patient_label") == "LUNG", F.col("patient_id"))).alias("lung_pts"),
            F.countDistinct(F.when(F.col("patient_label") == "HN", F.col("patient_id"))).alias("hn_pts"),
            F.countDistinct(F.when(F.col("patient_label") == "DUAL", F.col("patient_id"))).alias("dual_pts"),
            F.countDistinct("patient_id").alias("total_pts"))
       .withColumn("segment", F.when((F.col("lung_pts") > 0) & (F.col("hn_pts") > 0), "BOTH")
                                .when(F.col("lung_pts") > 0, "LUNG_ONLY")
                                .when(F.col("hn_pts") > 0, "HN_ONLY")
                                .when(F.col("dual_pts") > 0, "UNRESOLVED_DUAL")
                                .otherwise("NO_DX"))
       .cache())
hcp.groupBy("segment").count().orderBy("segment").show()

## 3. DDD → OPAL → account
OPAL is wide: `0` = HCP JJ_ID, `1`..`5` = parent at each level, `L2_L3` = L2 if present else L3.
It comes from `cdp.d_provider_hierarchy_pd` already filtered to ONC relationships.
First: at which OPAL level do the DDD `jnj_id`s appear? Then map each DDD id to an L2/L3 account.

### What this cell does: understand OPAL and find where the DDD accounts sit in it
- First it shows DDD units by product. DDD contains Rybrevant, Rybrevant Faspro **and Lazcluze**; Lazcluze is kept separate in the account table.
- Reads the OPAL columns as text:
  - `0` = doctor J&J ID;
  - `1`–`5` = parent at each level;
  - `L2_L3` = the account to use (L2 if it exists, else L3).
- **OPAL quality check**: number of rows and distinct doctors, how many rows have an L2, how many only have an L3, how many have no account, and whether `L2_L3` really equals "L2 else L3".
- Builds a list of every ID in OPAL with the level(s) it appears at.
- Looks up each **DDD `jnj_id`** in that list and shows the level it is found at (expected: L1 = outlet), with the units sold. `NOT_IN_OPAL` means the DDD account can't be placed.

In [ ]:
o = opal.select(*[F.col(f"`{c}`").cast("string").alias(c) for c in ["0", "1", "2", "3", "4", "5", "L2_L3"]],
                F.col("child_lvl").cast("int").alias("child_lvl"))
o = o.select(*[clean(c).alias(c) for c in ["0", "1", "2", "3", "4", "5", "L2_L3"]], "child_lvl")

# Quality of OPAL itself
o.agg(F.count("*").alias("rows"), F.countDistinct("0").alias("distinct_hcps"),
      F.sum(F.col("2").isNotNull().cast("int")).alias("rows_with_L2"),
      F.sum((F.col("2").isNull() & F.col("3").isNotNull()).cast("int")).alias("rows_L3_only"),
      F.sum(F.col("L2_L3").isNull().cast("int")).alias("rows_no_account"),
      F.sum((F.coalesce("2", "3") != F.col("L2_L3")).cast("int")).alias("L2_L3_disagrees")).show()

# Every OPAL id with its level
opal_ids = None
for lvl in ["0", "1", "2", "3", "4", "5"]:
    part = o.filter(F.col(lvl).isNotNull()).select(F.col(lvl).alias("jj_id"), F.lit("L" + lvl).alias("level"))
    opal_ids = part if opal_ids is None else opal_ids.unionByName(part)
opal_ids = opal_ids.distinct().groupBy("jj_id").agg(F.concat_ws("+", F.sort_array(F.collect_set("level"))).alias("opal_levels"))

ddd_c = ddd.select(F.col("jnj_id").cast("string").alias("jj_id"), F.to_date("caln_dt").alias("week"),
                   "prod_lst_id", F.col("prod_nm"), F.col("equiv_uns").cast("double").alias("equiv_uns"),
                   F.when(F.upper("prod_nm").contains("FASPRO"), "RYBREVANT_FASPRO")
                    .when(F.upper("prod_nm").contains("RYBREVANT"), "RYBREVANT")
                    .when(F.upper("prod_nm").contains("LAZCLUZE"), "LAZCLUZE").alias("product"))
ddd_c.groupBy("prod_nm", "product").agg(F.count("*").alias("rows"), F.round(F.sum("equiv_uns"), 1).alias("equiv_uns")).show()
(ddd_c.groupBy("jj_id").agg(F.sum("equiv_uns").alias("equiv_uns"))
      .join(opal_ids, "jj_id", "left").fillna({"opal_levels": "NOT_IN_OPAL"})
      .groupBy("opal_levels").agg(F.count("*").alias("ddd_ids"), F.round(F.sum("equiv_uns"), 1).alias("equiv_uns"))
      .orderBy(F.desc("ddd_ids")).show())

### What this cell does: give every DDD account its L2/L3 account
- Each DDD `jnj_id` gets the account it belongs to, in this order of preference:
  1. it is an **L1 outlet** (OPAL column `1`) → take the `L2_L3` value on those OPAL rows (the most common one if there are several);
  2. it is itself an **L2** account → use it as is;
  3. it is itself an **L3** account → use it as is.
- The result `ddd_acct` is the DDD sales table with an `account_id` added.
- Shows how many DDD IDs and what % of units were mapped by each route, and how much stayed **UNMAPPED**.

In [ ]:
# DDD id -> account. Priority: (a) id is an L1 outlet -> most frequent L2_L3 on its OPAL rows
#                              (b) id is itself an L2 account, (c) id is itself an L3 account
via_l1 = (o.filter(F.col("1").isNotNull() & F.col("L2_L3").isNotNull())
           .groupBy(F.col("1").alias("jj_id"), F.col("L2_L3").alias("account_id")).count()
           .withColumn("rk", F.row_number().over(Window.partitionBy("jj_id").orderBy(F.desc("count"), "account_id")))
           .filter("rk = 1").select("jj_id", "account_id", F.lit("L1_OUTLET").alias("map_via"), F.lit(1).alias("prio")))
via_l2 = o.filter(F.col("2").isNotNull()).select(F.col("2").alias("jj_id"), F.col("2").alias("account_id"),
                                                 F.lit("IS_L2").alias("map_via"), F.lit(2).alias("prio")).distinct()
via_l3 = o.filter(F.col("3").isNotNull()).select(F.col("3").alias("jj_id"), F.col("3").alias("account_id"),
                                                 F.lit("IS_L3").alias("map_via"), F.lit(3).alias("prio")).distinct()
ddd_map = (via_l1.unionByName(via_l2).unionByName(via_l3)
           .withColumn("rk", F.row_number().over(Window.partitionBy("jj_id").orderBy("prio")))
           .filter("rk = 1").drop("rk", "prio"))

ddd_acct = ddd_c.join(ddd_map, "jj_id", "left").cache()
(ddd_acct.groupBy(F.coalesce("map_via", F.lit("UNMAPPED")).alias("map_via"))
         .agg(F.countDistinct("jj_id").alias("ddd_ids"), F.round(F.sum("equiv_uns"), 1).alias("equiv_uns"))
         .withColumn("pct_units", F.round(100 * F.col("equiv_uns") / F.sum("equiv_uns").over(Window.partitionBy()), 1))
         .show())

## 4. HCP → account through DMN_CUST
Claims know doctors by **NPI** (`npi_id` in dmn_cust); OPAL and DDD use **J&J IDs** (`jnj_id` in dmn_cust).
DMN_CUST holds both doctors and organizations, so it can also link the **billing / pharmacy NPI** on a claim to a DDD account.

### What this cell does: look inside dmn_cust (only displays results)
- Shows what kinds of customers are in the table: `cust_super_typ_nm` / `cust_typ_nm` (doctor vs organization, etc.).
- **Check only:** `cust_stat_nm` should be only "Active" and `end_dt` only 2099-12-31, because upstream already applied those filters. If other values show up, the table was not pre-filtered.
- Counts the rows, rows that have an NPI, distinct NPIs and distinct J&J IDs.

In [ ]:
if dmn is not None:
    for c in ["cust_super_typ_nm", "cust_typ_nm", "cust_stat_nm"]:
        dmn.groupBy(c).count().orderBy(F.desc("count")).show(15, False)
    dmn.groupBy(F.to_date("end_dt").alias("end_dt")).count().orderBy(F.desc("count")).show(5)   # check: expect only 2099-12-31
    dmn.agg(F.count("*").alias("rows"),
            F.sum(F.col("npi_id").isNotNull().cast("int")).alias("rows_with_npi"),
            F.countDistinct("npi_id").alias("distinct_npi"),
            F.countDistinct("jnj_id").alias("distinct_jnj_id")).show()

### What this cell does: build the NPI → J&J ID lookup and check coverage
- Takes `npi_id` (NPI) and `jnj_id` (J&J ID) from dmn_cust, plus type and specialty (`prim_spclt_nm`).
- If one NPI has several J&J IDs, it keeps the one with the **latest `efftv_dt`** (the business rule from the data-sources file).
- Prints the coverage for the doctors in our claims:
  - how many claim NPIs are found in dmn_cust (and as what customer type);
  - how many of those J&J IDs exist in OPAL (column `0`);
  - how many NPIs had more than one J&J ID.

In [ ]:
if dmn is not None:
    d = (dmn.select(clean("npi_id").alias("npi"), F.col("jnj_id").cast("string").alias("hcp_jj_id"),
                    "cust_super_typ_nm", "cust_typ_nm", F.col("prim_spclt_nm").alias("specialty"),
                    F.to_date("efftv_dt").alias("efftv_dt"))
            .filter("npi IS NOT NULL AND hcp_jj_id IS NOT NULL"))
    # one J&J ID per NPI. Business rule: if one NPI has several jnj_ids, keep the one with the latest efftv_dt
    npi_jj = (d.withColumn("n_jj_ids_for_npi", F.size(F.collect_set("hcp_jj_id").over(Window.partitionBy("npi"))))
               .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(
                   F.desc_nulls_last("efftv_dt"), "hcp_jj_id")))
               .filter("rk = 1").drop("rk")
               .cache())

    # Coverage: how many claim doctors are found in dmn_cust, and are they in OPAL?
    claim_npis = claims.filter("npi IS NOT NULL").select("npi").distinct()
    cov = claim_npis.join(npi_jj, "npi", "left")
    (cov.withColumn("dmn_match", F.when(F.col("hcp_jj_id").isNull(), "NOT_IN_DMN_CUST")
                                  .otherwise(F.coalesce("cust_typ_nm", F.lit("?"))))
        .groupBy("dmn_match").count().orderBy(F.desc("count")).show(truncate=False))
    n_npi = claim_npis.count()
    n_jj = cov.filter("hcp_jj_id IS NOT NULL").count()
    n_opal = cov.join(o.select(F.col("0").alias("hcp_jj_id")).distinct(), "hcp_jj_id").count()
    n_multi = cov.filter("n_jj_ids_for_npi > 1").count()
    print(f"claim NPIs {n_npi:,} | found in dmn_cust {n_jj:,} ({100*n_jj/n_npi:.1f}%) | "
          f"J&J ID found in OPAL {n_opal:,} ({100*n_opal/n_npi:.1f}%) | NPIs with >1 J&J ID {n_multi:,}")
else:
    npi_jj = None
    claim_npis = claims.select("npi").filter("npi IS NOT NULL").distinct()
    print("claims NPIs:", claim_npis.count())
    print("NPIs found directly as OPAL HCP ids  :", claim_npis.join(o.select(F.col("0").alias("npi")).distinct(), "npi").count())
    print("-> DMN_CUST is required to link HCPs to accounts.")

### What this cell does: try the second route, claim organization → DDD account
- Medical claims have a **billing NPI** (usually the hospital or clinic); pharmacy claims have a **pharmacy NPI**.
- Looks those NPIs up in dmn_cust to get their J&J ID, then checks:
  - is that J&J ID a buyer in **DDD** (`in_ddd`)?
  - at which **OPAL** level does it appear?
- If many billing NPIs land directly on DDD accounts, we can link claims to sales without going through the doctor at all, as a cross-check of route 1.

In [ ]:
# Second route: billing NPI (medical) / pharmacy NPI (Rx) -> dmn_cust -> J&J ID -> is it a DDD account / in OPAL?
if npi_jj is not None:
    org = (claims.filter("org_npi IS NOT NULL").groupBy("source", "org_npi")
                 .agg(F.countDistinct("patient_id").alias("patients")))
    org_m = (org.join(npi_jj.select(F.col("npi").alias("org_npi"), F.col("hcp_jj_id").alias("org_jj_id"),
                                    F.col("cust_typ_nm").alias("org_type")), "org_npi", "left")
                .join(ddd_c.select(F.col("jj_id").alias("org_jj_id")).distinct().withColumn("in_ddd", F.lit(True)),
                      "org_jj_id", "left")
                .join(opal_ids.select(F.col("jj_id").alias("org_jj_id"), "opal_levels"), "org_jj_id", "left")
                .fillna({"in_ddd": False}))
    (org_m.groupBy("source", F.col("org_jj_id").isNotNull().alias("found_in_dmn"), "in_ddd", "opal_levels")
          .agg(F.count("*").alias("org_npis"), F.sum("patients").alias("patients"))
          .orderBy("source", F.desc("org_npis")).show(30, False))

### What this cell does: put each doctor on an account
- Path: doctor NPI → J&J ID (DMN_CUST) → OPAL column `0` → `L2_L3` account.
- A doctor linked to several accounts is split evenly (e.g. 2 accounts → 0.5 each), so nobody is counted twice.
- Prints the coverage:
  - how many claim doctors got a J&J ID;
  - how many reached an L2/L3 account.
- Shows the **HCP universe by specialty**: how many Lung-only / H&N-only / Both doctors in each specialty.

In [ ]:
if npi_jj is not None:
    hcp_opal = (o.filter(F.col("L2_L3").isNotNull())
                 .select(F.col("0").alias("hcp_jj_id"), F.col("L2_L3").alias("account_id")).distinct()
                 .withColumn("weight", 1.0 / F.count("*").over(Window.partitionBy("hcp_jj_id"))))   # multi-account HCPs split 1/n
    hcp_acct = (hcp.join(npi_jj.select("npi", "hcp_jj_id", "specialty", "cust_typ_nm"), "npi", "left")
                   .join(hcp_opal, "hcp_jj_id", "left").cache())

    n = hcp.count()
    m = hcp_acct.filter("hcp_jj_id IS NOT NULL").select("npi").distinct().count()
    a = hcp_acct.filter("account_id IS NOT NULL").select("npi").distinct().count()
    print(f"claims HCPs {n:,} | with J&J ID {m:,} ({100*m/n:.1f}%) | with L2/L3 account {a:,} ({100*a/n:.1f}%)")

    # HCP universe by specialty x segment
    (hcp_acct.select("npi", "specialty", "segment").distinct()
             .groupBy(F.coalesce("specialty", F.lit("UNKNOWN")).alias("specialty"))
             .pivot("segment", ["LUNG_ONLY", "HN_ONLY", "BOTH", "UNRESOLVED_DUAL", "NO_DX"]).count()
             .fillna(0).orderBy(F.desc("LUNG_ONLY")).show(25, False))

## 5. Account table: DDD sales + HCP mix

### What this cell does: build the final account table
- **Sales side** (always runs): for each L2/L3 account, adds up DDD units:
  - Rybrevant, Faspro, and `total_units` = Rybrevant + Faspro (the units to split);
  - Lazcluze separately (lung-only, not split).
  - Also shows the first and last week of sales.
- **Doctor side** (needs dmn_cust): for each account, the number of Lung-only, H&N-only and Both doctors, plus their Lung and H&N patients.
- Each account is tagged:
  - **MIXED**: has both Lung and H&N doctors;
  - **LUNG_ONLY** or **HN_ONLY**;
  - **NO_LINKED_HCPS**: sales but no known doctors.
- Shows what % of units comes from each type, then the top 20 accounts by units.
- The `accounts` table is the main Phase 1 output for the Phase 2 split.

In [ ]:
acct_sales = (ddd_acct.filter("account_id IS NOT NULL")
              .groupBy("account_id")
              .agg(F.round(F.sum(F.when(F.col("product") == "RYBREVANT", F.col("equiv_uns"))), 3).alias("rybrevant_units"),
                   F.round(F.sum(F.when(F.col("product") == "RYBREVANT_FASPRO", F.col("equiv_uns"))), 3).alias("faspro_units"),
                   F.round(F.sum(F.when(F.col("product").isin(RYB_PRODUCTS), F.col("equiv_uns"))), 3).alias("total_units"),
                   F.round(F.sum(F.when(F.col("product") == "LAZCLUZE", F.col("equiv_uns"))), 3).alias("lazcluze_units"),
                   F.countDistinct("jj_id").alias("ddd_ids"),
                   F.min("week").alias("first_week"), F.max("week").alias("last_week")))

if npi_jj is not None:
    acct_hcp = (hcp_acct.filter("account_id IS NOT NULL").groupBy("account_id")
                .agg(*[F.sum(F.when(F.col("segment") == s, F.col("weight")).otherwise(0)).alias(s.lower() + "_hcps")
                       for s in ["LUNG_ONLY", "HN_ONLY", "BOTH"]],
                     F.sum(F.col("lung_pts") * F.col("weight")).alias("lung_pts"),
                     F.sum(F.col("hn_pts") * F.col("weight")).alias("hn_pts")))
    accounts = (acct_sales.join(acct_hcp, "account_id", "full").fillna(0)
                .withColumn("account_mix",
                    F.when(((F.col("lung_only_hcps") + F.col("both_hcps")) > 0) & ((F.col("hn_only_hcps") + F.col("both_hcps")) > 0), "MIXED")
                     .when((F.col("lung_only_hcps") + F.col("both_hcps")) > 0, "LUNG_ONLY")
                     .when(F.col("hn_only_hcps") > 0, "HN_ONLY")
                     .otherwise("NO_LINKED_HCPS")))
    (accounts.groupBy("account_mix").agg(F.count("*").alias("accounts"), F.round(F.sum("total_units"), 1).alias("units"))
             .withColumn("pct_units", F.round(100 * F.col("units") / F.sum("units").over(Window.partitionBy()), 1)).show())
else:
    accounts = acct_sales

accounts.orderBy(F.desc("total_units")).show(20, False)

## 6. ChargeMaster check: national NSCLC / HNSCC share vs claims
ChargeMaster columns are strings; blocks `_1/_2/_3` are not yet confirmed (likely IV / Faspro / ?).
`NSCLC/HNSCC/Other` can sum > 1 (overlapping), `*_1` sums to 1 (exclusive).

### What this cell does: compare with the ChargeMaster national numbers
- ChargeMaster stores numbers as text, so they are converted to numbers first.
- Shows the cadences available (annual, monthly, ...) and the yearly rows: patients, encounters, and the NSCLC / HNSCC / Other shares.
- Builds a **month-by-month** view and puts it next to the same months from our claims:
  - ChargeMaster HNSCC share (`hnscc`, `hnscc_1`);
  - claims: administrations, and how many had an H&N vs Lung diagnosis (`claims_hn_share`).
- Only Rybrevant / Faspro administrations are counted (Lazcluze excluded).
- If the two H&N shares are close, the claims-based labels are believable.

In [ ]:
num = lambda c: F.regexp_replace(F.col(f"`{c}`").cast("string"), ",", "").cast("double")
cm_c = cm.select(F.col("`Time Frame (Sorter)`").alias("tf"), F.col("`Time Frame Descriptive`").alias("tf_desc"),
                 "Cadence", num("# Patients").alias("patients"), num("# Encounters").alias("encounters"),
                 num("# Encounters_1").alias("enc_1"), num("# Encounters_2").alias("enc_2"), num("# Encounters_3").alias("enc_3"),
                 num("NSCLC").alias("nsclc"), num("HNSCC").alias("hnscc"), num("Other").alias("other"),
                 num("NSCLC_1").alias("nsclc_1"), num("HNSCC_1").alias("hnscc_1"), num("Other_1").alias("other_1"))
cm_c.groupBy("Cadence").count().show(truncate=False)
cm_c.filter(F.col("tf_desc").startswith("CY-")).orderBy("tf").show(truncate=False)

monthly_cm = (cm_c.filter(F.col("tf_desc").startswith("MN-"))
              .withColumn("month", F.to_date(F.regexp_extract("tf_desc", r"(\d{4}-\d{2}-\d{2})", 1)))
              .select("month", "encounters", "hnscc", "hnscc_1"))

# Same month view from claims: administrations by the dx on the claim
adm = claims.filter(F.col("product").isin(RYB_PRODUCTS)).dropDuplicates(["patient_id", "service_dt", "product"])
monthly_claims = (adm.groupBy("month")
                  .agg(F.count("*").alias("claims_admins"),
                       F.sum((F.col("claim_label") == "HN").cast("int")).alias("claims_hn_admins"),
                       F.sum((F.col("claim_label") == "LUNG").cast("int")).alias("claims_lung_admins"))
                  .withColumn("claims_hn_share", F.round(F.col("claims_hn_admins") / F.col("claims_admins"), 4)))
monthly_cm.join(monthly_claims, "month", "full").orderBy("month").show(40, False)

## Summary of the link status
- Claims → patient → HCP label: **works** (sections 2).
- DDD → OPAL → L2/L3 account: **works** (section 3); check the share of units that maps.
- HCP → account: through `dmn_cust` (`npi_id` → `jnj_id`) then OPAL `0` → `L2_L3` (section 4). Check the coverage line.
- ChargeMaster: national check of the H&N share; compare with `claims_hn_share` (section 6).